# 88 — Site Coordinate Registry (deduplicated)

Deduplicates site keys and prefers rows with permit IDs and manual coordinates.

In [ ]:
import os, re, json, hashlib, platform, sys, math
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {"project": "AirQuality26", "step": step_name, "created_at_utc": utcnow(), "platform": {"python": sys.version, "platform": platform.platform()}, "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()}, "artifacts": [], "notes": []}

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def haversine_miles(lat1, lon1, lat2, lon2):
    vals = [lat1, lon1, lat2, lon2]
    if any(pd.isna(vals)):
        return np.nan
    r = 3958.7613
    p1, p2 = math.radians(float(lat1)), math.radians(float(lat2))
    dphi = math.radians(float(lat2) - float(lat1))
    dlmb = math.radians(float(lon2) - float(lon1))
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return 2*r*math.asin(math.sqrt(a))

In [ ]:
step = "88_site_coordinate_registry"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

run_cfg = load_yaml(CONFIGS / "run.yml")
geo_cfg = load_yaml(CONFIGS / "site_geospatial_registry.yml")
priority, _ = safe_read_csv(OUTPUTS / "85_national_priority_matrix" / "national_priority_matrix.csv")

manual = []
for key, spec in (geo_cfg.get("sites", {}) or {}).items():
    manual.append({
        "site_key": str(key),
        "permit_id": str(spec.get("permit_id", "")).strip(),
        "facility_name": spec.get("facility_name", key),
        "lat": spec.get("lat"),
        "lon": spec.get("lon"),
        "nearest_town": spec.get("nearest_town", ""),
        "nearest_city": spec.get("nearest_city", ""),
        "control_site": bool(spec.get("control_site", False)),
        "site_type": spec.get("site_type", "unknown"),
        "_source_rank": 3,
    })
manual_df = pd.DataFrame(manual)

run_sites = pd.DataFrame(run_cfg.get("sites", []))
run_rows = []
if not run_sites.empty:
    run_sites["site_id"] = run_sites.get("id", pd.Series(dtype=str)).astype(str)
    run_sites["site_name"] = run_sites.get("name", pd.Series(dtype=str)).astype(str)
    for _, r in run_sites.iterrows():
        run_rows.append({
            "site_key": r.get("site_id"),
            "permit_id": "",
            "facility_name": r.get("site_name"),
            "lat": r.get("lat", np.nan),
            "lon": r.get("lon", np.nan),
            "nearest_town": "",
            "nearest_city": "",
            "control_site": str(r.get("role","")).lower() == "control",
            "site_type": "configured_repo_site",
            "_source_rank": 2,
        })
run_df = pd.DataFrame(run_rows)

priority_df = pd.DataFrame()
if not priority.empty:
    priority_df = priority[["permit_id","facility_name"]].drop_duplicates().copy()
    priority_df["site_key"] = priority_df["permit_id"].astype(str).str.lower()
    priority_df["lat"] = np.nan
    priority_df["lon"] = np.nan
    priority_df["nearest_town"] = ""
    priority_df["nearest_city"] = ""
    priority_df["control_site"] = False
    priority_df["site_type"] = "priority_permit"
    priority_df["_source_rank"] = 1

combined = pd.concat([manual_df, run_df, priority_df], ignore_index=True, sort=False)
combined["site_key"] = combined["site_key"].astype(str).replace("nan","").replace("", np.nan)
combined["site_key"] = combined["site_key"].fillna(
    combined["facility_name"].astype(str).str.lower().str.replace(r"[^a-z0-9]+","_", regex=True).str.strip("_")
)
combined["_has_permit"] = combined["permit_id"].astype(str).str.len().gt(0).astype(int)
combined["_has_coords"] = (combined["lat"].notna() & combined["lon"].notna()).astype(int)
combined = combined.sort_values(["site_key","_source_rank","_has_permit","_has_coords"], ascending=[True, False, False, False])
combined = combined.groupby("site_key", as_index=False).first()
combined = combined.drop(columns=[c for c in ["_source_rank","_has_permit","_has_coords"] if c in combined.columns])

combined.to_csv(out / "site_coordinate_registry.csv", index=False)
write_json(out / "site_coordinate_registry_summary.json", {
    "rows": int(len(combined)),
    "with_coordinates": int(combined["lat"].notna().sum()),
    "controls": int(combined["control_site"].fillna(False).astype(bool).sum()),
})

manifest = manifest_base(step, [CONFIGS / "site_geospatial_registry.yml"])
add_artifact(manifest, out / "site_coordinate_registry.csv")
add_artifact(manifest, out / "site_coordinate_registry_summary.json")
write_json(out / "manifest.json", manifest)
display(combined.head(20))